In [18]:
import torch 
from torch import nn
from torch.nn import functional as F

class Residual(nn.Module):
    
    def __init__(self, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        
        # main branch: 2 convolutional layers
        self.conv1=nn.LazyConv2d(num_channels,kernel_size=3,
                                 padding=1,stride=strides)
        self.conv2=nn.LazyConv2d(num_channels,kernel_size=3,
                                 padding=1,stride=1)
        
        # skip connection 
        if use_1x1conv:
            self.conv3 = nn.LazyConv2d(num_channels,kernel_size=1,
                                       stride=strides)
        else:
            self.conv3 = None
            
        self.bn1 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm2d()
            
    def forward(self,X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3 is not None:
            X = self.conv3(X)
        return F.relu(Y + X)

In [19]:
arch = ((2, 64), (2, 128), (2, 256), (2, 512))

In [20]:
import d2l


class ResNet(nn.Module):
    def b1(self):
        return nn.Sequential(
            nn.LazyConv2d(64, kernel_size=7, stride=2, padding=3),
            nn.LazyBatchNorm2d(),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

    def block(self, num_residuals, num_channels, first_block=False):
        """1 stage contains `num_residuals` residual blocks."""
        blk = []

        for i in range(num_residuals):
            if i == 0 and not first_block:
                blk.append(Residual(num_channels, use_1x1conv=True, strides=2))
            else:
                blk.append(Residual(num_channels))
        return nn.Sequential(*blk)

    def __init__(self, arch, lr=0.1, num_classes=10):
        super(ResNet, self).__init__()
        self.net = nn.Sequential(self.b1())
        for i, b in enumerate(arch):
            self.net.add_module(f"b{i+2}", self.block(*b, first_block=(i == 0)))
        self.net.add_module(
            "last",
            nn.Sequential(
                nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.LazyLinear(num_classes)
            ),
        )
    
    def forward(self, x):
        return self.net(x)

In [21]:
class ResNet18(ResNet):
    def __init__(self, lr=0.1, num_classes=10):
        super().__init__(((2, 64), (2, 128), (2, 256), (2, 512)),
                         lr, num_classes)


model = ResNet18()
# Test with dummy input
X = torch.randn(1, 1, 96, 96)
output = model(X)
print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print("ResNet18 model created successfully!")

Input shape: torch.Size([1, 1, 96, 96])
Output shape: torch.Size([1, 10])
ResNet18 model created successfully!
